# Step 1: Protocol and cross-validation folds
### Neeraj, MMAI 894 (router fine-tuning)

**Goal:** set up the fixed rules every later experiment follows, and draw the cross-validation folds once.

**Why this comes first:** with only 160 labelled chats to develop on, one lucky or unlucky split can change a result by several points. So we (1) check we are using the right dataset file, (2) lock the 40 test chats away until the very end, and (3) cut the 160 chats into 5 folds *once*, so every method is compared on exactly the same folds.

**What you need to do:** run all cells (Internet must be ON; no GPU is needed for this notebook, so choose "None" under Accelerator to save your GPU hours). At the end, download the files listed in the last cell.

The rules themselves are written in `docs/EVAL_PROTOCOL.md`; the code is in `src/northstar/finetune/protocol.py` and `make_folds.py`.

## Step 0: Get the code
Kaggle starts with an empty folder, so we download the repo from GitHub (it is public, so no login is needed). Using the repo as the single source of code means the notebook only *runs* things; the logic lives in files that are version-controlled and reviewed.

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

# Always start from a fresh copy of the latest code. Kaggle keeps files between runs in one session,
# so an old copy of the repo would silently hide newer commits.
%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}          # download the repo (public, so no login is needed)

%cd {REPO_DIR}
sys.path.insert(0, "src")                    # lets us write: from northstar.finetune import protocol
importlib.invalidate_caches()                # forget what Python saw in the folder on an earlier run
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]                    # drop stale copies of our modules from earlier runs

assert os.path.exists("src/northstar/finetune/protocol.py"), "protocol.py is missing: has the latest code been pushed to GitHub?"
!git log --oneline -3                        # shows exactly which version of the code is running

## Step 1: Check we have the right dataset
The team's dataset file is identified by its SHA-256 fingerprint. `load_all()` refuses to run if the file differs by even one character, so nobody can silently train on a different version. (We found earlier that an older copy differed in three escalation labels, which is exactly the kind of drift this check prevents.)

`pool()` returns only the 160 train and validation chats. The 40 test chats are not in it, so ordinary code cannot even see them.

In [ ]:
import json
import pandas as pd
from northstar.finetune import protocol as P

df = P.load_all()                       # raises an error if the CSV is not the exact team file (hash check)
print("All chats:", len(df), "| split sizes:", df["split"].value_counts().to_dict())
print("Chats that need a person:", int(df["escalation_label"].sum()), "| customers:", df["customer_id"].nunique())

pool = P.pool(df)                       # train + validation only; the test chats are not in here
print("Pool used for development:", len(pool), "chats from", pool["customer_id"].nunique(), "customers")

## Step 2: The three inputs a router can read
A router can be given different amounts of the chat. We will compare all three later, on identical chats:

| Variant | Text | Why it matters |
|---|---|---|
| `first_message` | the customer's first message | what a real router sees before any reply exists |
| `customer_turns` | the customer's first and last messages | what the original notebook used |
| `full_conversation` | customer, agent, customer | what the team's pipeline passes today; the middle line is an AI-written agent reply that may hint at the answer |

Below: the same chat in each form, and how long each form is.

In [ ]:
example = pool[pool["language"] == "en"].iloc[[0]]          # one English chat, kept as a 1-row table
for variant in P.INPUT_VARIANTS:
    text = P.input_text(example, variant).iloc[0]
    print(f"--- {variant} ({len(text)} characters)\n{text}\n")

# How long is each form across the whole pool? (characters, not tokens)
lengths = pd.DataFrame({v: P.input_text(pool, v).str.len() for v in P.INPUT_VARIANTS})
lengths.describe().loc[["mean", "50%", "max"]].round(0)

## Step 3: Draw the folds
**Idea of cross-validation:** cut the 160 chats into 5 groups (folds). Train on 4, test on the 5th, rotate five times. Every chat is scored once by a model that never saw it, so we get 160 honest predictions instead of 30.

**Two rules for the cut:**
1. **Group by customer.** One customer often wrote several chats in similar words. If some of their chats were in training and others in testing, the model would be tested on things it effectively memorised. So all chats of one customer stay in one fold.
2. **Keep the folds alike.** Each fold should have a similar share of every request type, of chats that need a person, of attack chats, of French chats, and a similar size. A first attempt with a plain random grouped split came out lopsided: one fold had 4 French chats and another had 15, because there are only 79 customers to place. So `make_folds.py` uses a small search that moves customers between folds until the mix is even. It looks at labels only, never at any model's results.

The folds are saved to a file and never redrawn, because fold-splitting routines can give different folds on different library versions. Three repeats are saved; the first is the default.

In [ ]:
from northstar.finetune import make_folds

make_folds.main()      # writes data/folds/folds_v1.csv and folds_v1.meta.json, and prints a balance summary

## Step 4: Check the folds
Three hard checks (the cell stops with an error if any fails): every chat has a fold, no customer is split across folds, and the folds contain exactly the 160 pool chats (no test chat).

Then the balance table, built from the saved file: for each fold, the number of chats, how many need a person, how many are attacks, how many are French, and the smallest and largest count of any request type. Read it like this: the rows within one repeat should look similar to each other, and `fewest_of_a_type` should be at least 1 (no fold is missing a request type).

In [ ]:
folds = pd.read_csv(P.FOLDS_PATH)
meta = json.load(open(P.FOLDS_PATH.parent / "folds_v1.meta.json"))

for rep in range(3):
    col = f"rep{rep}"
    assert folds[col].notna().all(), "a chat has no fold"
    assert (folds.groupby("customer_id")[col].nunique() == 1).all(), "a customer is split across folds"
assert set(folds["id"]) == set(pool["id"]), "the folds must contain exactly the 160 pool chats"
print("All checks passed. Search settings:", {k: meta[k] for k in ("method", "restarts", "iterations")})

rows = []
for rep in range(3):
    for k in range(P.N_FOLDS):
        s = folds[folds[f"rep{rep}"] == k]
        per_type = s["request_type"].value_counts().reindex(P.REQUEST_TYPES, fill_value=0)
        rows.append(dict(rep=rep, fold=k, chats=len(s), need_person=int(s["escalation_label"].sum()),
                         attacks=int(s["adversarial"].sum()), french=int((s["language"] == "fr").sum()),
                         fewest_of_a_type=int(per_type.min()), most_of_a_type=int(per_type.max())))
balance = pd.DataFrame(rows)
balance

In [ ]:
import matplotlib.pyplot as plt

# First repeat only: do the folds look alike on the things that matter?
b0 = balance[balance["rep"] == 0].set_index("fold")
ax = b0[["chats", "need_person", "attacks", "french"]].plot(kind="bar", figsize=(8, 4))
ax.set_title("Fold balance, repeat 0 (bars should look similar across folds)")
ax.set_ylabel("number of chats")
plt.tight_layout()
os.makedirs("/kaggle/working/results_01", exist_ok=True)
plt.savefig("/kaggle/working/results_01/fold_balance.png", dpi=150)
plt.show()

## Step 5: The test lock, demonstrated
The 40 test chats are scored once, at the very end. `test_set()` refuses to return them unless the caller passes `final=True` **and** the environment variable `NORTHSTAR_ALLOW_TEST=1` is set, and every opening is written to a log file. Below we try to open them the ordinary way and confirm it is refused.

In [ ]:
try:
    P.test_set(df)
    print("PROBLEM: the test chats opened without permission")
except PermissionError as err:
    print("Refused, as intended:", err)

## Step 6: Save the results
Copy the files we need into `/kaggle/working/results_01/`, and print a fingerprint of the folds file so we can confirm later that the committed copy is identical.

**After the run:** open the notebook's *Output* tab, download `folds_v1.csv`, `folds_v1.meta.json`, `fold_balance.csv` and `fold_balance.png`, and put the two `folds_v1` files into `data/folds/` in the repo. Then commit them.

## What this notebook shows, and its limits
- **Shows:** the dataset check passes, the test chats are locked, and the folds are valid (hard checks) and similar to each other (balance table and chart; read them, do not trust this sentence).
- **Limits:**
  - Only 79 customers exist in the pool, so folds can never be perfectly even.
  - Chats from *different* customers can still be near-duplicates (the team found one similar towing pair); they are not grouped.
  - All chats were written by one generator, so cross-validation tells us how the model does on more chats *like these*, not on real customers.
  - 160 chats still means wide uncertainty on every result; later steps report intervals, not single numbers.

In [ ]:
import hashlib
import shutil

out = "/kaggle/working/results_01"
shutil.copy(P.FOLDS_PATH, f"{out}/folds_v1.csv")
shutil.copy(P.FOLDS_PATH.parent / "folds_v1.meta.json", f"{out}/folds_v1.meta.json")
balance.to_csv(f"{out}/fold_balance.csv", index=False)

print("SHA-256 of folds_v1.csv:", hashlib.sha256(open(P.FOLDS_PATH, "rb").read()).hexdigest())
print("Files to download:", sorted(os.listdir(out)))